Importation des données

In [1]:
import pandas as pd

df = pd.read_csv("Envoy_Air_pre_Cleaned.csv")

# Afficher les premières lignes
display(df.head())

# Vérifier le nombre de lignes et de colonnes
print("Dimensions du dataset :", df.shape)

,FlightDate,Origin,Dest,CRSDepTime,ArrDelayMinutes,Distance,Year,Month,DayofMonth,DayOfWeek,...,DestCityMarketID,DestCityName,DestState,DestStateFips,DestStateName,DestWac,DepTimeBlk,CRSArrTime,ArrDel15,ArrTimeBlk
0,2018-10-26,IAH,ORD,1137,0.0,925.0,2018,10,26,5,...,30977,"Chicago, IL",IL,17,Illinois,41,1100-1159,1420,0.0,1400-1459
1,2018-10-27,IAH,ORD,1137,4.0,925.0,2018,10,27,6,...,30977,"Chicago, IL",IL,17,Illinois,41,1100-1159,1420,0.0,1400-1459
2,2018-10-28,IAH,ORD,1137,0.0,925.0,2018,10,28,7,...,30977,"Chicago, IL",IL,17,Illinois,41,1100-1159,1420,0.0,1400-1459
3,2018-10-29,IAH,ORD,1137,0.0,925.0,2018,10,29,1,...,30977,"Chicago, IL",IL,17,Illinois,41,1100-1159,1420,0.0,1400-1459
4,2018-10-30,IAH,ORD,1137,0.0,925.0,2018,10,30,2,...,30977,"Chicago, IL",IL,17,Illinois,41,1100-1159,1420,0.0,1400-1459


Dimensions du dataset : (1033979, 29)


In [2]:
# Nom et type de chaque variable
display(pd.DataFrame({
    "Variable": df.columns,
    "Type": df.dtypes.astype(str).values
}))

,Variable,Type
0,FlightDate,object
1,Origin,object
2,Dest,object
3,CRSDepTime,int64
4,ArrDelayMinutes,float64
5,Distance,float64
6,Year,int64
7,Month,int64
8,DayofMonth,int64
9,DayOfWeek,int64


Variables redondante à retirer

In [3]:
# Suppression des variables redondantes
colonnes_a_supprimer = [
    "OriginCityName",
    "DestCityName",
    "OriginStateName",
    "DestStateName"
]

# Supprimer uniquement les colonnes présentes
colonnes_a_supprimer = [
    col for col in colonnes_a_supprimer if col in df.columns
]

df = df.drop(columns=colonnes_a_supprimer)

# Afficher les résultats
print("Variables supprimées :")
for col in colonnes_a_supprimer:
    print("-", col)

print("\nNombre de variables supprimées :", len(colonnes_a_supprimer))
print("Nombre de variables restantes :", len(df.columns))

Variables supprimées :
- OriginCityName
- DestCityName
- OriginStateName
- DestStateName

Nombre de variables supprimées : 4
Nombre de variables restantes : 25


Préparer les variables explicatives et les cibles

In [4]:
# Variables cibles
y_classification = df["ArrDel15"]
y_regression = df["ArrDelayMinutes"]

# Variables explicatives
X = df.drop(columns=["ArrDel15", "ArrDelayMinutes", "FlightDate"])

# Vérifier les dimensions
print("Variables explicatives (X) :", X.shape)
print("Cible classification :", y_classification.shape)
print("Cible régression :", y_regression.shape)

Variables explicatives (X) : (1033979, 22)
Cible classification : (1033979,)
Cible régression : (1033979,)


Vérification les types de variables

In [5]:
# Afficher le type de chaque variable
display(pd.DataFrame({
    "Variable": X.columns,
    "Type": X.dtypes.astype(str).values
}))

# Compter les variables par type
print("\nNombre de variables numériques :",
      len(X.select_dtypes(include="number").columns))

print("Nombre de variables catégorielles :",
      len(X.select_dtypes(include=["object", "category", "string"]).columns))

,Variable,Type
0,Origin,object
1,Dest,object
2,CRSDepTime,int64
3,Distance,float64
4,Year,int64
5,Month,int64
6,DayofMonth,int64
7,DayOfWeek,int64
8,Flight_Number_Marketing_Airline,int64
9,OriginAirportID,int64



Nombre de variables numériques : 16
Nombre de variables catégorielles : 6


Séparer les données pour la classification

In [6]:
# Variables explicatives : sans les deux cibles
X_class = df.drop(
    columns=["ArrDel15", "ArrDelayMinutes", "FlightDate"]
)

# Cible de classification uniquement
y_class = df["ArrDel15"].astype(int)

# Séparation chronologique 80 % / 20 %
limite = int(len(df) * 0.80)

X_class_train = X_class.iloc[:limite].copy()
X_class_test = X_class.iloc[limite:].copy()

y_class_train = y_class.iloc[:limite].copy()
y_class_test = y_class.iloc[limite:].copy()

# Vérifier les dimensions
print("X entraînement :", X_class_train.shape)
print("X test :", X_class_test.shape)
print("y entraînement :", y_class_train.shape)
print("y test :", y_class_test.shape)

X entraînement : (827183, 22)
X test : (206796, 22)
y entraînement : (827183,)
y test : (206796,)


Vérification des variables textuelles

In [7]:
# Identifier les variables catégorielles
variables_textuelles = X_class_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

print("Variables à encoder :")
for col in variables_textuelles:
    print("-", col)

Variables à encoder :
- Origin
- Dest
- OriginState
- DestState
- DepTimeBlk
- ArrTimeBlk


Encoder les variables

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Identifier les variables catégorielles et numériques
variables_textuelles = X_class_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

variables_numeriques = X_class_train.select_dtypes(
    include=["number"]
).columns.tolist()

# Vérifier le nombre de catégories par variable
print("Nombre de catégories par variable :")

for col in variables_textuelles:
    print(f"{col} : {X_class_train[col].nunique()} catégories")

# Préparer les variables numériques
traitement_numerique = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Préparer les variables catégorielles
traitement_categoriel = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combiner les traitements
preprocesseur = ColumnTransformer(transformers=[
    ("numeriques", traitement_numerique, variables_numeriques),
    ("categoriels", traitement_categoriel, variables_textuelles)
])

# Apprendre les transformations sur les données d'entraînement
X_train_encode = preprocesseur.fit_transform(X_class_train)

# Appliquer les mêmes transformations aux données de test
X_test_encode = preprocesseur.transform(X_class_test)

print("\nDimensions après encodage :")
print("X_train :", X_train_encode.shape)
print("X_test :", X_test_encode.shape)

Nombre de catégories par variable :
Origin : 187 catégories
Dest : 187 catégories
OriginState : 46 catégories
DestState : 46 catégories
DepTimeBlk : 19 catégories
ArrTimeBlk : 19 catégories

Dimensions après encodage :
X_train : (827183, 520)
X_test : (206796, 520)


Créer l'echantillon de 100 000 vols

In [9]:
# Prendre les 100 000 premières lignes de l'entraînement
# (les données sont supposées déjà triées par date)
X_echantillon = X_train_encode[:100000]
y_echantillon = y_class_train.iloc[:100000]

print("Dimensions de X :", X_echantillon.shape)
print("Nombre de cibles :", len(y_echantillon))

Dimensions de X : (100000, 520)
Nombre de cibles : 100000


Entrainement classification 

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier

# 1. Régression logistique
modele_logistique = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)
modele_logistique.fit(X_echantillon, y_echantillon)

# 2. Arbre de décision
modele_arbre = DecisionTreeClassifier(
    max_depth=15,
    class_weight="balanced",
    random_state=42
)
modele_arbre.fit(X_echantillon, y_echantillon)

# 3. Random Forest
modele_foret = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=10,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)
modele_foret.fit(X_echantillon, y_echantillon)

# 4. SVM
modele_svm = LinearSVC(
    class_weight="balanced",
    max_iter=2000,
    random_state=42
)
modele_svm.fit(X_echantillon, y_echantillon)

# 5. KNN
modele_knn = KNeighborsClassifier(
    n_neighbors=5,
    n_jobs=-1
)
modele_knn.fit(X_echantillon, y_echantillon)

print("Les 5 modèles sont entraînés sur 100 000 vols.")

c:\Users\segui\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Les 5 modèles sont entraînés sur 100 000 vols.


Comparer les modèles de classification

In [11]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

modeles = {
    "Régression logistique": modele_logistique,
    "Arbre de décision": modele_arbre,
    "Random Forest": modele_foret,
    "SVM": modele_svm,
    "KNN": modele_knn
}

resultats = []

for nom, modele in modeles.items():
    print(f"Évaluation : {nom}...")

    predictions = modele.predict(X_test_encode)

    resultats.append({
        "Modèle": nom,
        "Accuracy": accuracy_score(y_class_test, predictions),
        "Precision": precision_score(
            y_class_test, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_class_test, predictions, zero_division=0
        ),
        "F1-score": f1_score(
            y_class_test, predictions, zero_division=0
        )
    })

df_resultats = pd.DataFrame(resultats)

display(
    df_resultats.sort_values(
        by="F1-score", ascending=False
    ).round(4)
)

Évaluation : Régression logistique...
Évaluation : Arbre de décision...
Évaluation : Random Forest...
Évaluation : SVM...
Évaluation : KNN...


,Modèle,Accuracy,Precision,Recall,F1-score
0,Régression logistique,0.5956,0.2044,0.4652,0.2841
3,SVM,0.5740,0.1963,0.4752,0.2779
2,Random Forest,0.6442,0.2074,0.3769,0.2676
1,Arbre de décision,0.6017,0.1875,0.3926,0.2538
4,KNN,0.7937,0.2082,0.0700,0.1048
